In [1]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
from utils import plot_y_yhat,error_metric
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.ensemble import HistGradientBoostingRegressor
from catboost import CatBoostRegressor, Pool

In [2]:
train=pd.read_csv('data/train_data.csv')
test=pd.read_csv('data/test_data.csv')
sample_submission=pd.read_csv('data/sample_submission.csv')

In [6]:
# Use only non-censored patients 
train_non_censored = train[train['Censored'] == 0].copy()

# Remove any patients with missing SurvivalTime (safety check)
train_clean = train_non_censored.dropna(subset=['SurvivalTime'])

# Select features (keep ALL columns even with missing values)
feature_columns = [col for col in train_clean.columns 
                   if col not in ['SurvivalTime', 'Censored', 'id']]
X = train_clean[feature_columns]
y = train_clean['SurvivalTime']

print("Decision Tree - Data Overview:")
print(f"- Patients: {len(X)}")
print(f"- Features: {len(feature_columns)}")
print(f"- Columns with missing values: {X.isnull().any().sum()}")

# TRAIN-TEST SPLIT + CROSS-VALIDATION
#Split initial 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# FINDING BEST PARAMETERS 

param_grid = {
    'max_depth': [5, 7, 10, 15, None],  # Arbres plus profonds
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2', 0.8]  # Important pour la robustesse
}
# GridSearch fait automatiquement la cross-validation
grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error'
)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
print(f"Best parameters: {best_params}")
best_dt = DecisionTreeRegressor(**best_params, random_state=42)


#  FIT FINAL MODEL
best_dt.fit(X_train, y_train)
y_pred = best_dt.predict(X_test)
bt_error = error_metric(y_test,y_pred,c=0)
print(f"Decision Trees Error: {bt_error:.4f} years")



Decision Tree - Data Overview:
- Patients: 161
- Features: 7
- Columns with missing values: 3
Best parameters: {'max_depth': 5, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 2}
Decision Trees Error: 3.5755 years


In [7]:
#  HistGradientBoostingRegressor
hgbr = HistGradientBoostingRegressor(random_state=42)
cv_scores = cross_val_score(
    hgbr,
    X_train, y_train,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

cv_cMSE = -cv_scores  # Convert to positive cMSE
print(f"Cross-validation scores: {cv_cMSE}")
print(f"Mean cMSE: {cv_cMSE.mean():.4f}")
print(f"Std cMSE: {cv_cMSE.std():.4f}")

#  TRAINING FINAL SUR TOUT LE TRAIN SET
hgbr.fit(X_train, y_train)

# ÉVALUATION SUR TEST SET
y_pred_hgbr = hgbr.predict(X_test)
hgbr_error = error_metric(y_test, y_pred_hgbr, c=0)
print(f"HistGradientBoostingRegressor Error: {bt_error:.4f} years")

Cross-validation scores: [3.66298135 4.49174944 3.0511632  3.57933121 4.7850629 ]
Mean cMSE: 3.9141
Std cMSE: 0.6344
HistGradientBoostingRegressor Error: 3.5755 years


In [8]:
# 1. PRÉPARATION DES DONNÉES
all_data = train.dropna(subset=['SurvivalTime']).copy()
survival_features = [col for col in all_data.columns 
                    if col not in ['SurvivalTime', 'Censored', 'id']]

X_surv = all_data[survival_features]
y_surv = all_data['SurvivalTime']
censor = all_data['Censored']
print(X_surv.isnull().sum())
# Format pour CatBoost Survival
y_catboost = np.column_stack([y_surv, censor])
print(y_catboost)
X_train_aft, X_test_aft, y_train_aft, y_test_aft = train_test_split(
    X_surv, y_catboost, test_size=0.2, random_state=42
)
catboost_final = CatBoostRegressor(
    iterations=1000,
    loss_function='SurvivalAft:dist=Normal',
    eval_metric='SurvivalAft',
    random_seed=42,
    verbose=0
)

# Entraînement final
train_pool = Pool(X_train_aft, y_train_aft)
test_pool = Pool(X_test_aft, y_test_aft)
catboost_final.fit(
    train_pool,
    eval_set=test_pool
)

# Évaluation sur test set
y_pred = catboost_final.predict(X_test_aft, prediction_type='Exponent')
print(f"Prédictions: {y_pred[:10]}")  # Temps de survie réels
y_true_test = y_test_aft[:, 0]
c_test = y_test_aft[:, 1]
def interval_mae(y_true, y_pred):
    mae = np.where((y_true <= y_pred),
                   0,
                  np.abs(y_true-y_pred),
                            ) 
    return mae.mean()

catboost_error = error_metric(y_true_test,y_pred,c_test)
print(f"Test set cMSE: {catboost_error:.4f}")

Age                   0
Gender                0
Stage                 0
GeneticRisk          47
TreatmentType         0
ComorbidityIndex     30
TreatmentResponse    16
dtype: int64
[[ 4.2  0. ]
 [ 4.7  1. ]
 [ 3.5  1. ]
 [ 2.3  0. ]
 [ 2.6  0. ]
 [ 6.4  0. ]
 [ 4.1  0. ]
 [ 9.1  0. ]
 [ 3.3  1. ]
 [ 3.   0. ]
 [ 4.7  0. ]
 [ 2.7  0. ]
 [ 1.7  0. ]
 [ 3.6  0. ]
 [ 5.1  0. ]
 [ 5.3  1. ]
 [ 4.1  0. ]
 [ 7.5  1. ]
 [ 4.8  0. ]
 [ 1.3  0. ]
 [ 3.8  1. ]
 [ 2.   0. ]
 [ 6.8  0. ]
 [ 4.3  1. ]
 [ 5.2  0. ]
 [ 1.3  0. ]
 [ 2.6  0. ]
 [ 3.3  1. ]
 [ 6.4  0. ]
 [ 1.9  1. ]
 [ 3.4  1. ]
 [ 5.8  0. ]
 [ 5.6  0. ]
 [ 2.5  1. ]
 [ 2.6  1. ]
 [ 5.8  0. ]
 [ 0.5  0. ]
 [ 5.2  0. ]
 [ 2.8  0. ]
 [ 5.2  1. ]
 [ 1.9  0. ]
 [ 3.4  0. ]
 [ 3.9  1. ]
 [ 6.8  0. ]
 [ 4.1  0. ]
 [ 4.   1. ]
 [ 2.9  0. ]
 [ 7.4  1. ]
 [ 4.   0. ]
 [ 5.7  0. ]
 [ 3.6  1. ]
 [ 7.3  0. ]
 [ 3.9  1. ]
 [ 6.8  0. ]
 [ 4.3  0. ]
 [ 4.4  1. ]
 [ 3.5  0. ]
 [ 7.5  0. ]
 [ 2.7  1. ]
 [ 4.1  1. ]
 [ 1.3  0. ]
 [ 1.4  1. ]
 [ 5.5  0. ]
